# 🐍 Módulo 00: Python desde cero

Variables, texto, listas, diccionarios, decisiones, bucles, funciones, errores y clases: la base para leer y escribir cualquier código.

**PandasQuest** · versión web interactiva: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m00)

**Cómo funciona:**
1. Ejecuta la celda ⚙️ de abajo (carga las tablas y el corrector).
2. En cada ejercicio escribe tu código y guarda la respuesta en `resultado`.
3. Ejecuta la celda: `comprobar()` te dirá si está bien. 🎯

¿Atascado? `ver_solucion("id")` muestra la solución (¡pero inténtalo antes! 😉)

In [ ]:
#@title ⚙️ 1) Ejecuta esta celda primero (carga datos y corrector) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    return False, f"Tu resultado es {u!r}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usa & (y), | (o) y pon cada condición entre paréntesis."),
    ("KeyError", "Revisa que el nombre de la columna esté bien escrito (mayúsculas, tildes, espacios)."),
    ("NameError", "Estás usando una variable que no existe. ¿Un error de tipeo?"),
    ("SyntaxError", "Hay un error de sintaxis: revisa paréntesis, comillas y que la línea `resultado = ...` esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este objeto. Revisa el nombre."),
    ("TypeError", "Algún tipo de dato no encaja (¿texto mezclado con números?)."),
    ("IndentationError", "Revisa la sangría (espacios al inicio de la línea)."),
    ("MergeError", "Revisa las columnas que usas para unir (on / left_on / right_on)."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n'
_SOL_B64 = 'eyJweTAxIjogeyJzb2x1dGlvbiI6ICJjaXVkYWQgPSAnQ8OzcmRvYmEnXG5oYWJpdGFudGVzID0gMV81MDBfMDAwXG5yZXN1bHRhZG8gPSBoYWJpdGFudGVzIiwgImNoZWNrIjogeyJjdXN0b20iOiAiY2l1ZGFkID09ICdDw7NyZG9iYScgYW5kIHJlc3VsdGFkbyA9PSAxNTAwMDAwIiwgImN1c3RvbV9tc2ciOiAiUmV2aXNhIHF1ZSBgY2l1ZGFkYCBzZWEgJ0PDs3Jkb2JhJyB5IHF1ZSBgcmVzdWx0YWRvYCB0ZW5nYSBsb3MgaGFiaXRhbnRlcyAoMTUwMDAwMCkuIn0sICJ4cCI6IDEwfSwgInB5MDIiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IFt0eXBlKDcpLl9fbmFtZV9fLCB0eXBlKDcuNSkuX19uYW1lX18sIHR5cGUoJzcnKS5fX25hbWVfXywgdHlwZShUcnVlKS5fX25hbWVfX10iLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gWydpbnQnLCAnZmxvYXQnLCAnc3RyJywgJ2Jvb2wnXSIsICJjdXN0b21fbXNnIjogIkRlYmVuIHNlciA0IG5vbWJyZXMgZGUgdGlwbyBlbiBvcmRlbjogaW50LCBmbG9hdCwgc3RyLCBib29sLiJ9LCAieHAiOiAxMH0sICJweTAzIjogeyJzb2x1dGlvbiI6ICJkdXJhY2lvbiA9IDUwMFxubWludXRvcyA9IGR1cmFjaW9uIC8vIDYwXG5zZWd1bmRvcyA9IGR1cmFjaW9uICUgNjBcbnJlc3VsdGFkbyA9IFttaW51dG9zLCBzZWd1bmRvc10iLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gWzgsIDIwXSIsICJjdXN0b21fbXNnIjogIjUwMCBzZWd1bmRvcyBzb24gOCBtaW51dG9zIHkgMjAgc2VndW5kb3MuIFVzYSAvLyB5ICUuIn0sICJ4cCI6IDEwfSwgInB5MDQiOiB7InNvbHV0aW9uIjogInByZWNpb19hID0gJzEyNTAuNTAnXG5wcmVjaW9fYiA9ICc3NDkuNTAnXG5yZXN1bHRhZG8gPSBmbG9hdChwcmVjaW9fYSkgKyBmbG9hdChwcmVjaW9fYikiLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJpc2luc3RhbmNlKHJlc3VsdGFkbywgKGludCwgZmxvYXQpKSBhbmQgYWJzKHJlc3VsdGFkbyAtIDIwMDAuMCkgPCAxZS05IiwgImN1c3RvbV9tc2ciOiAiRWwgcmVzdWx0YWRvIGRlYmUgc2VyIGVsIG7Dum1lcm8gMjAwMC4wLiDCv0NvbnZlcnRpc3RlIGxvcyBkb3MgdGV4dG9zIGNvbiBmbG9hdCgpPyJ9LCAieHAiOiAyMH0sICJweTA1IjogeyJzb2x1dGlvbiI6ICJhbHVtbm8gPSAnTHVjaWFubydcbm5vdGEgPSA4Ljc1MTJcbnJlc3VsdGFkbyA9IGYne2FsdW1ub30gYXByb2LDsyBjb24ge25vdGE6LjJmfSBwdW50b3MnIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvID09ICdMdWNpYW5vIGFwcm9iw7MgY29uIDguNzUgcHVudG9zJyIsICJjdXN0b21fbXNnIjogIkVsIHRleHRvIGRlYmUgc2VyIGV4YWN0YW1lbnRlOiBMdWNpYW5vIGFwcm9iw7MgY29uIDguNzUgcHVudG9zIn0sICJ4cCI6IDEwfSwgInB5MDYiOiB7InNvbHV0aW9uIjogInN1Y2lvID0gJyAgIHNBTiBtaWd1ZWwgZGUgVFVDVU3DgU4gICdcbnJlc3VsdGFkbyA9IHN1Y2lvLnN0cmlwKCkudGl0bGUoKSIsICJjaGVjayI6IHsiY3VzdG9tIjogInJlc3VsdGFkbyA9PSAnU2FuIE1pZ3VlbCBEZSBUdWN1bcOhbiciLCAiY3VzdG9tX21zZyI6ICJEZWJlIHF1ZWRhciAnU2FuIE1pZ3VlbCBEZSBUdWN1bcOhbic6IHVzYSAuc3RyaXAoKSB5IC50aXRsZSgpLiJ9LCAieHAiOiAxMH0sICJweTA3IjogeyJzb2x1dGlvbiI6ICJjb2RpZ28gPSAnQVJHLTIwMjQtMDA3MzEnXG5wYWlzID0gY29kaWdvWzozXVxuc2VyaWUgPSBjb2RpZ29bLTU6XVxucmVzdWx0YWRvID0gW3BhaXMsIHNlcmllXSIsICJjaGVjayI6IHsiY3VzdG9tIjogInJlc3VsdGFkbyA9PSBbJ0FSRycsICcwMDczMSddIiwgImN1c3RvbV9tc2ciOiAiRGViZSBxdWVkYXIgWydBUkcnLCAnMDA3MzEnXS4gUmVjdWVyZGE6IFs6M10geSBbLTU6XS4ifSwgInhwIjogMjB9LCAicHkwOCI6IHsic29sdXRpb24iOiAiZW1haWwgPSAnbHVjaWFuby5yb3NhbGVzQGRhdG9zLmNvbSdcbnVzdWFyaW8gPSBlbWFpbC5zcGxpdCgnQCcpWzBdXG5yZXN1bHRhZG8gPSAnXycuam9pbih1c3VhcmlvLnNwbGl0KCcuJykpIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvID09ICdsdWNpYW5vX3Jvc2FsZXMnIiwgImN1c3RvbV9tc2ciOiAiRGViZSBxdWVkYXIgJ2x1Y2lhbm9fcm9zYWxlcycuIn0sICJ4cCI6IDIwfSwgInB5MDkiOiB7InNvbHV0aW9uIjogInRlbXBlcmF0dXJhcyA9IFsxOC41LCAyMi4wLCAyNS41LCAxOS4wLCAzMC4wXVxucmVzdWx0YWRvID0gW21heCh0ZW1wZXJhdHVyYXMpLCBtaW4odGVtcGVyYXR1cmFzKSwgc3VtKHRlbXBlcmF0dXJhcykgLyBsZW4odGVtcGVyYXR1cmFzKV0iLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJsZW4ocmVzdWx0YWRvKSA9PSAzIGFuZCByZXN1bHRhZG9bMF0gPT0gMzAuMCBhbmQgcmVzdWx0YWRvWzFdID09IDE4LjUgYW5kIGFicyhyZXN1bHRhZG9bMl0gLSAyMy4wKSA8IDFlLTkiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiBzZXIgMyB2YWxvcmVzOiBtw6F4aW1hICgzMC4wKSwgbcOtbmltYSAoMTguNSkgeSBwcm9tZWRpbyAoMjMuMCkuIn0sICJ4cCI6IDEwfSwgInB5MTAiOiB7InNvbHV0aW9uIjogIm5vdGFzID0gWzcsIDIsIDEwLCA2LCA4XVxubm90YXMuYXBwZW5kKDkpXG5ub3Rhcy5yZW1vdmUoMilcbm5vdGFzLnNvcnQocmV2ZXJzZT1UcnVlKVxucmVzdWx0YWRvID0gbm90YXMiLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gWzEwLCA5LCA4LCA3LCA2XSIsICJjdXN0b21fbXNnIjogIkxhIGxpc3RhIGZpbmFsIGRlYmUgc2VyIFsxMCwgOSwgOCwgNywgNl0uIn0sICJ4cCI6IDIwfSwgInB5MTEiOiB7InNvbHV0aW9uIjogImRpbWVuc2lvbmVzID0gKDE1MDAsIDEyKVxuZmlsYXMsIGNvbHVtbmFzID0gZGltZW5zaW9uZXNcbnJlc3VsdGFkbyA9IGZpbGFzICogY29sdW1uYXMiLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJmaWxhcyA9PSAxNTAwIGFuZCBjb2x1bW5hcyA9PSAxMiBhbmQgcmVzdWx0YWRvID09IDE4MDAwIiwgImN1c3RvbV9tc2ciOiAiRGVzZW1wYXF1ZXRhIGNvbiBgZmlsYXMsIGNvbHVtbmFzID0gZGltZW5zaW9uZXNgIHkgbXVsdGlwbGljYSAocmVzdWx0YWRvOiAxODAwMCkuIn0sICJ4cCI6IDIwfSwgInB5MTIiOiB7InNvbHV0aW9uIjogInByb2R1Y3RvID0geydub21icmUnOiAnTW9uaXRvcicsICdwcmVjaW8nOiAxMDAwfVxucHJvZHVjdG9bJ3ByZWNpbyddID0gMTEwMFxucHJvZHVjdG9bJ3N0b2NrJ10gPSA4XG5yZXN1bHRhZG8gPSBwcm9kdWN0byIsICJjaGVjayI6IHsiY3VzdG9tIjogInJlc3VsdGFkbyA9PSB7J25vbWJyZSc6ICdNb25pdG9yJywgJ3ByZWNpbyc6IDExMDAsICdzdG9jayc6IDh9IiwgImN1c3RvbV9tc2ciOiAiRWwgZGljY2lvbmFyaW8gZGViZSBxdWVkYXIgeydub21icmUnOiAnTW9uaXRvcicsICdwcmVjaW8nOiAxMTAwLCAnc3RvY2snOiA4fS4ifSwgInhwIjogMTB9LCAicHkxMyI6IHsic29sdXRpb24iOiAidmVudGFzID0geydTb2bDrWEnOiAxMjAwLCAnVG9tw6FzJzogOTUwLCAnVmFsZXJpYSc6IDE0MzAsICdBbmRyw6lzJzogODcwfVxucmVzdWx0YWRvID0gW3N1bSh2ZW50YXMudmFsdWVzKCkpLCBtYXgodmVudGFzLCBrZXk9dmVudGFzLmdldCldIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvID09IFs0NDUwLCAnVmFsZXJpYSddIiwgImN1c3RvbV9tc2ciOiAiRGViZW4gc2VyIGVsIHRvdGFsICg0NDUwKSB5IGVsIG1lam9yIHZlbmRlZG9yICgnVmFsZXJpYScpLiJ9LCAieHAiOiAyMH0sICJweTE0IjogeyJzb2x1dGlvbiI6ICJlbmVybyA9IFsnQW5hJywgJ0JydW5vJywgJ0NhcmxhJywgJ0RpZWdvJywgJ0FuYSddXG5mZWJyZXJvID0gWydCcnVubycsICdFbGVuYScsICdEaWVnbyddXG5yZXN1bHRhZG8gPSBzb3J0ZWQoc2V0KGVuZXJvKSAtIHNldChmZWJyZXJvKSkiLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gWydBbmEnLCAnQ2FybGEnXSIsICJjdXN0b21fbXNnIjogIkRlYmVuIHF1ZWRhciBbJ0FuYScsICdDYXJsYSddLCBvcmRlbmFkb3MgeSBzaW4gcmVwZXRpci4ifSwgInhwIjogMjB9LCAicHkxNSI6IHsic29sdXRpb24iOiAibm90YSA9IDdcbmlmIG5vdGEgPj0gODpcbiAgICByZXN1bHRhZG8gPSAnUHJvbW9jaW9uYSdcbmVsaWYgbm90YSA+PSA2OlxuICAgIHJlc3VsdGFkbyA9ICdBcHJ1ZWJhJ1xuZWxzZTpcbiAgICByZXN1bHRhZG8gPSAnUmVjdXBlcmEnIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvID09ICdBcHJ1ZWJhJyBhbmQgJ2lmJyBpbiBfcHFfY29kZSBhbmQgJ2VsaWYnIGluIF9wcV9jb2RlIiwgImN1c3RvbV9tc2ciOiAiQ29uIG5vdGEgNyBkZWJlIGRhciAnQXBydWViYScuIFVzYSBpZiAvIGVsaWYgLyBlbHNlLiJ9LCAieHAiOiAxMH0sICJweTE2IjogeyJzb2x1dGlvbiI6ICJnYXN0byA9IDYyMDAwXG5wcmVtaXVtID0gRmFsc2VcbnByaW1lcmFfY29tcHJhID0gVHJ1ZVxucmVzdWx0YWRvID0gKGdhc3RvID4gNTAwMDAgYW5kIHByZW1pdW0pIG9yIHByaW1lcmFfY29tcHJhIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvIGlzIFRydWUgYW5kICdvcicgaW4gX3BxX2NvZGUgYW5kICdhbmQnIGluIF9wcV9jb2RlIiwgImN1c3RvbV9tc2ciOiAiQ29tYmluYSBsYXMgY29uZGljaW9uZXMgY29uIGBhbmRgIHkgYG9yYC4gQ29uIGVzdG9zIGRhdG9zIGVsIHJlc3VsdGFkbyBlcyBUcnVlLiJ9LCAieHAiOiAyMH0sICJweTE3IjogeyJzb2x1dGlvbiI6ICJ0b3RhbCA9IDBcbmZvciBuIGluIHJhbmdlKDIsIDEwMSwgMik6XG4gICAgdG90YWwgKz0gblxucmVzdWx0YWRvID0gdG90YWwiLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gMjU1MCBhbmQgJ2ZvcicgaW4gX3BxX2NvZGUiLCAiY3VzdG9tX21zZyI6ICJMYSBzdW1hIGRlIGxvcyBwYXJlcyBkZWwgMiBhbCAxMDAgZXMgMjU1MC4gVXNhIHVuIGJ1Y2xlIGZvci4ifSwgInhwIjogMTB9LCAicHkxOCI6IHsic29sdXRpb24iOiAicGFnb3MgPSBbODAwMCwgMTIwMDAsIDE1MDAwLCA5NTAwLCAxMDAwMCwgMTEwMDBdXG5hbHRvcyA9IFtdXG5mb3IgcCBpbiBwYWdvczpcbiAgICBpZiBwID4gMTAwMDA6XG4gICAgICAgIGFsdG9zLmFwcGVuZChwKVxucmVzdWx0YWRvID0gYWx0b3MiLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gWzEyMDAwLCAxNTAwMCwgMTEwMDBdIiwgImN1c3RvbV9tc2ciOiAiRGViZW4gcXVlZGFyIFsxMjAwMCwgMTUwMDAsIDExMDAwXSAoMTAwMDAgbm8gZXMgbWF5b3IgYSAxMDAwMCkuIn0sICJ4cCI6IDIwfSwgInB5MTkiOiB7InNvbHV0aW9uIjogImFob3JybyA9IDUwMDAwXG5tZXNlcyA9IDBcbndoaWxlIGFob3JybyA8IDEwMDAwMDpcbiAgICBhaG9ycm8gKj0gMS4wNVxuICAgIG1lc2VzICs9IDFcbnJlc3VsdGFkbyA9IG1lc2VzIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvID09IDE1IGFuZCAnd2hpbGUnIGluIF9wcV9jb2RlIiwgImN1c3RvbV9tc2ciOiAiQ29uIHVuIDUlIG1lbnN1YWwgaGFjZW4gZmFsdGEgMTUgbWVzZXMuIFVzYSB3aGlsZS4ifSwgInhwIjogMjB9LCAicHkyMCI6IHsic29sdXRpb24iOiAicHJvZHVjdG9zID0gWydMYXB0b3AnLCAnTW91c2UnLCAnTW9uaXRvciddXG5wcmVjaW9zID0gWzEyMDAsIDI1LCAzMDBdXG5jYW50aWRhZGVzID0gWzMsIDQwLCA1XVxucmVzdWx0YWRvID0ge31cbmZvciBwcm9kLCBwcmVjaW8sIGNhbnQgaW4gemlwKHByb2R1Y3RvcywgcHJlY2lvcywgY2FudGlkYWRlcyk6XG4gICAgcmVzdWx0YWRvW3Byb2RdID0gcHJlY2lvICogY2FudCIsICJjaGVjayI6IHsiY3VzdG9tIjogInJlc3VsdGFkbyA9PSB7J0xhcHRvcCc6IDM2MDAsICdNb3VzZSc6IDEwMDAsICdNb25pdG9yJzogMTUwMH0iLCAiY3VzdG9tX21zZyI6ICJEZWJlIHF1ZWRhciB7J0xhcHRvcCc6IDM2MDAsICdNb3VzZSc6IDEwMDAsICdNb25pdG9yJzogMTUwMH0uIn0sICJ4cCI6IDIwfSwgInB5MjEiOiB7InNvbHV0aW9uIjogInByZWNpb3MgPSBbMTAwLCAyNS41LCA4MCwgNDUsIDMwMF1cbnJlc3VsdGFkbyA9IFtyb3VuZChwICogMS4yMSwgMikgZm9yIHAgaW4gcHJlY2lvcyBpZiBwID4gNTBdIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvID09IFsxMjEuMCwgOTYuOCwgMzYzLjBdIGFuZCAnIGZvciAnIGluIF9wcV9jb2RlIGFuZCAnWycgaW4gX3BxX2NvZGUiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiBxdWVkYXIgWzEyMS4wLCA5Ni44LCAzNjMuMF0sIGhlY2hvIGNvbiB1bmEgbGlzdCBjb21wcmVoZW5zaW9uLiJ9LCAieHAiOiAzNX0sICJweTIyIjogeyJzb2x1dGlvbiI6ICJkZWYgcHJlY2lvX2ZpbmFsKHByZWNpbywgZGVzY3VlbnRvKTpcbiAgICByZXR1cm4gcHJlY2lvICogKDEgLSBkZXNjdWVudG8gLyAxMDApXG5cbnJlc3VsdGFkbyA9IHByZWNpb19maW5hbCgxNTAwMCwgMjApIiwgImNoZWNrIjogeyJjdXN0b20iOiAiYWJzKHJlc3VsdGFkbyAtIDEyMDAwKSA8IDFlLTkgYW5kIGFicyhwcmVjaW9fZmluYWwoMTAwMCwgNTApIC0gNTAwKSA8IDFlLTkgYW5kIGFicyhwcmVjaW9fZmluYWwoMjAwLCAwKSAtIDIwMCkgPCAxZS05IiwgImN1c3RvbV9tc2ciOiAicHJlY2lvX2ZpbmFsKDE1MDAwLCAyMCkgZGViZSBkZXZvbHZlciAxMjAwMC4gwr9Vc2FzdGUgcmV0dXJuPyJ9LCAieHAiOiAxMH0sICJweTIzIjogeyJzb2x1dGlvbiI6ICJkZWYgY29udmVydGlyKG1vbnRvLCBjb3RpemFjaW9uPTEwMDApOlxuICAgIHJldHVybiByb3VuZChtb250byAvIGNvdGl6YWNpb24sIDIpXG5cbnJlc3VsdGFkbyA9IFtjb252ZXJ0aXIoMjUwMDAwKSwgY29udmVydGlyKDI1MDAwMCwgY290aXphY2lvbj0xMjUwKV0iLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gWzI1MC4wLCAyMDAuMF0gYW5kIGNvbnZlcnRpcigxMDAwKSA9PSAxLjAiLCAiY3VzdG9tX21zZyI6ICJEZWJlIGRhciBbMjUwLjAsIDIwMC4wXS4gwr9QdXNpc3RlIGNvdGl6YWNpb249MTAwMCBjb21vIHZhbG9yIHBvciBkZWZlY3RvPyJ9LCAieHAiOiAyMH0sICJweTI0IjogeyJzb2x1dGlvbiI6ICJwcm9kdWN0b3MgPSBbKCdNb3VzZScsIDI1KSwgKCdMYXB0b3AnLCAxMjAwKSwgKCdNb25pdG9yJywgMzAwKSwgKCdUZWNsYWRvJywgNDUpXVxub3JkZW5hZG9zID0gc29ydGVkKHByb2R1Y3Rvcywga2V5PWxhbWJkYSBwOiBwWzFdLCByZXZlcnNlPVRydWUpXG5yZXN1bHRhZG8gPSBbcFswXSBmb3IgcCBpbiBvcmRlbmFkb3NdIiwgImNoZWNrIjogeyJjdXN0b20iOiAicmVzdWx0YWRvID09IFsnTGFwdG9wJywgJ01vbml0b3InLCAnVGVjbGFkbycsICdNb3VzZSddIGFuZCAnbGFtYmRhJyBpbiBfcHFfY29kZSIsICJjdXN0b21fbXNnIjogIkVsIG9yZGVuIGRlYmUgc2VyIExhcHRvcCwgTW9uaXRvciwgVGVjbGFkbywgTW91c2UuIFVzYSBzb3J0ZWQgY29uIGtleT1sYW1iZGEuIn0sICJ4cCI6IDM1fSwgInB5MjUiOiB7InNvbHV0aW9uIjogImVkYWRlc190ZXh0byA9IFsnMzQnLCAnMjgnLCAnbi9kJywgJzQ1JywgJycsICcxOSddXG5yZXN1bHRhZG8gPSBbXVxuZm9yIGUgaW4gZWRhZGVzX3RleHRvOlxuICAgIHRyeTpcbiAgICAgICAgcmVzdWx0YWRvLmFwcGVuZChpbnQoZSkpXG4gICAgZXhjZXB0IFZhbHVlRXJyb3I6XG4gICAgICAgIHBhc3MiLCAiY2hlY2siOiB7ImN1c3RvbSI6ICJyZXN1bHRhZG8gPT0gWzM0LCAyOCwgNDUsIDE5XSBhbmQgJ2V4Y2VwdCcgaW4gX3BxX2NvZGUiLCAiY3VzdG9tX21zZyI6ICJEZWJlbiBxdWVkYXIgWzM0LCAyOCwgNDUsIDE5XS4gVXNhIHRyeSAvIGV4Y2VwdCBWYWx1ZUVycm9yLiJ9LCAieHAiOiAyMH0sICJweTI2IjogeyJzb2x1dGlvbiI6ICJpbXBvcnQgc3RhdGlzdGljc1xuc3VlbGRvcyA9IFszMDAwMCwgMzUwMDAsIDQwMDAwLCA0NTAwMCwgNDUwMDAsIDU1MDAwLCA1MDAwMDBdXG5yZXN1bHRhZG8gPSBbc3RhdGlzdGljcy5tZWFuKHN1ZWxkb3MpLCBzdGF0aXN0aWNzLm1lZGlhbihzdWVsZG9zKSwgc3RhdGlzdGljcy5tb2RlKHN1ZWxkb3MpXSIsICJjaGVjayI6IHsiY3VzdG9tIjogImxlbihyZXN1bHRhZG8pID09IDMgYW5kIGFicyhyZXN1bHRhZG9bMF0gLSA3NTAwMDAgLyA3KSA8IDFlLTYgYW5kIHJlc3VsdGFkb1sxXSA9PSA0NTAwMCBhbmQgcmVzdWx0YWRvWzJdID09IDQ1MDAwIiwgImN1c3RvbV9tc2ciOiAiRGViZW4gc2VyIG1lZGlhICh+MTA3MTQyLjg2KSwgbWVkaWFuYSAoNDUwMDApIHkgbW9kYSAoNDUwMDApLiJ9LCAieHAiOiAxMH0sICJweTI3IjogeyJzb2x1dGlvbiI6ICJjbGFzcyBDdWVudGE6XG4gICAgZGVmIF9faW5pdF9fKHNlbGYsIHNhbGRvKTpcbiAgICAgICAgc2VsZi5zYWxkbyA9IHNhbGRvXG5cbiAgICBkZWYgZGVwb3NpdGFyKHNlbGYsIG1vbnRvKTpcbiAgICAgICAgc2VsZi5zYWxkbyArPSBtb250b1xuXG4gICAgZGVmIHJldGlyYXIoc2VsZiwgbW9udG8pOlxuICAgICAgICBpZiBtb250byA8PSBzZWxmLnNhbGRvOlxuICAgICAgICAgICAgc2VsZi5zYWxkbyAtPSBtb250b1xuXG5jdWVudGEgPSBDdWVudGEoMTAwMClcbmN1ZW50YS5kZXBvc2l0YXIoNTAwKVxuY3VlbnRhLnJldGlyYXIoMjAwMClcbmN1ZW50YS5yZXRpcmFyKDMwMClcbnJlc3VsdGFkbyA9IGN1ZW50YS5zYWxkbyIsICJjaGVjayI6IHsiY3VzdG9tIjogInJlc3VsdGFkbyA9PSAxMjAwIGFuZCBDdWVudGEoNTApLnNhbGRvID09IDUwIiwgImN1c3RvbV9tc2ciOiAiRWwgc2FsZG8gZmluYWwgZGViZSBzZXIgMTIwMDogMTAwMCArIDUwMCwgZWwgcmV0aXJvIGRlIDIwMDAgbm8gc2UgaGFjZSwgeSBzZSByZXRpcmFuIDMwMC4ifSwgInhwIjogMzV9fQ=='
_TABLAS = []

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todas las tablas limpias (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    if ok:
        from IPython.display import HTML, display
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">🎉 ¡Correcto! +' + str(info["xp"]) + ' XP · Sigue así 🐼</div>'))
    else:
        from IPython.display import HTML, display
        import html as _h
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">❌ Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_solucion(ej):
    print(_SOL[ej]["solution"])

reiniciar_datos()
print("✅ Listo." + (" Datos cargados: " + ", ".join(_TABLAS) if _TABLAS else " Ya puedes empezar."))


---
# 📌 Primeros pasos

## 1. Hola, Python
`Básico` · **+10 XP**

**Python** es el lenguaje número uno en ciencia de datos e inteligencia artificial. pandas, NumPy, scikit-learn, PyTorch… todo se escribe en Python. Antes de analizar datos, necesitas leer y escribir Python con soltura.

Una **variable** es un nombre que guarda un valor. Se crea con `=` (se lee "guarda en"):

```python
nombre = "Ana"        # texto (string): entre comillas
edad = 28             # número entero (int)
altura = 1.65         # número decimal (float)
print(nombre, edad)   # print muestra valores en pantalla -> Ana 28
```

Reglas para nombrar variables:
- Letras minúsculas, números y guion bajo: `precio_final`, `total_2024`.
- No pueden empezar con un número ni tener espacios.
- Python distingue mayúsculas: `edad` y `Edad` son variables distintas.

> 💡 En este curso, cada ejercicio se corrige mirando la variable `resultado`. Guarda siempre ahí tu respuesta.

### 🎯 Tu misión
Crea la variable `ciudad` con el texto `'Córdoba'` y la variable `habitantes` con el número `1_500_000` (el guion bajo solo facilita la lectura). Después guarda en `resultado` la variable `habitantes`.

<details><summary>💡 Pista</summary>

`habitantes = 1_500_000  y luego  resultado = habitantes`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

# Escribe tu código aquí 👇
ciudad = 
habitantes = 
resultado = 

comprobar("py01")

## 2. Tipos de datos
`Básico` · **+10 XP**

Cada valor en Python tiene un **tipo**. Los cuatro básicos son:

| Tipo | Qué guarda | Ejemplo |
|---|---|---|
| `int` | números enteros | `42`, `-7` |
| `float` | números decimales | `3.14`, `0.5` |
| `str` | texto | `"hola"`, `'Python'` |
| `bool` | verdadero o falso | `True`, `False` |

La función `type()` te dice el tipo de cualquier valor:

```python
type(42)        # <class 'int'>
type("42")      # <class 'str'>  ← ¡con comillas es texto!
type(4.0)       # <class 'float'>
```

Para quedarte solo con el nombre del tipo: `type(42).__name__` devuelve `'int'`.

> 💼 **En el trabajo:** la mitad de los errores al limpiar datos son de tipo: números que llegaron como texto (`"1200"`) o fechas que llegaron como `str`.

### 🎯 Tu misión
Guarda en `resultado` una **lista** con los nombres de los tipos de estos cuatro valores, en este orden: `7`, `7.5`, `'7'`, `True`. Usa `type(valor).__name__`.

<details><summary>💡 Pista</summary>

`[type(7).__name__, type(7.5).__name__, type('7').__name__, type(True).__name__]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = [type(7).__name__, ]

comprobar("py02")

## 3. La calculadora de Python
`Básico` · **+10 XP**

Python calcula como una calculadora, con algunos operadores extra muy útiles:

| Operador | Qué hace | Ejemplo | Resultado |
|---|---|---|---|
| `+ - * /` | suma, resta, multiplicación, división | `7 / 2` | `3.5` |
| `//` | división entera (sin decimales) | `7 // 2` | `3` |
| `%` | resto de la división (módulo) | `7 % 2` | `1` |
| `**` | potencia | `2 ** 3` | `8` |

Se respetan las reglas matemáticas: primero potencias, luego `* / // %`, y al final `+ -`. Usa paréntesis para cambiar el orden.

```python
minutos = 135
horas = minutos // 60      # 2
resto = minutos % 60       # 15
```

> 💡 `%` sirve para saber si un número es par: `n % 2 == 0`.

### 🎯 Tu misión
Un video dura `500` segundos. Calcula cuántos **minutos completos** tiene y cuántos **segundos sobran**. Guarda en `resultado` una lista `[minutos, segundos]`.

<details><summary>💡 Pista</summary>

`minutos = duracion // 60   y   segundos = duracion % 60`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

duracion = 500
minutos = 
segundos = 
resultado = [minutos, segundos]

comprobar("py03")

## 4. Convertir tipos
`Intermedio` · **+20 XP**

Muchas veces un número llega como texto (por ejemplo, desde un formulario o un archivo). Con texto no se puede calcular: `"10" + "5"` da `"105"`, porque **pega** los textos.

Para convertir entre tipos se usan funciones con el nombre del tipo:

```python
int("42")       # 42     texto -> entero
float("3.5")    # 3.5    texto -> decimal
str(100)        # "100"  número -> texto
int(9.99)       # 9      corta los decimales (no redondea)
round(9.99, 1)  # 10.0   redondea a 1 decimal
```

> ⚠️ `int("3.5")` da error: primero pásalo a `float` y después a `int`.

### 🎯 Tu misión
Los precios llegaron como texto. Conviértelos a números decimales y guarda en `resultado` la **suma** de los dos.

<details><summary>💡 Pista</summary>

`float(precio_a) + float(precio_b)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

precio_a = '1250.50'
precio_b = '749.50'
resultado = precio_a + precio_b   # ⚠️ esto pega textos

comprobar("py04")

---
# 📌 Texto (strings)

## 5. f-strings: texto con variables
`Básico` · **+10 XP**

Las **f-strings** son la forma moderna de meter variables dentro de un texto. Se escribe una `f` antes de las comillas y cada variable va entre llaves `{}`:

```python
nombre = "Ana"
edad = 28
f"{nombre} tiene {edad} años"          # 'Ana tiene 28 años'
f"El año que viene tendrá {edad + 1}"  # dentro de {} puede ir cualquier cálculo
```

También sirven para dar formato a los números:

```python
precio = 1234.5678
f"${precio:.2f}"     # '$1234.57'   -> 2 decimales
f"{precio:,.0f}"     # '1,235'      -> separador de miles, sin decimales
f"{0.256:.1%}"       # '25.6%'      -> como porcentaje
```

> 💼 **En el trabajo:** los reportes, los mensajes de log y los títulos de gráficos se arman con f-strings.

### 🎯 Tu misión
Con las variables dadas, guarda en `resultado` exactamente el texto: `Luciano aprobó con 8.75 puntos` (la nota con **2 decimales**).

<details><summary>💡 Pista</summary>

`f'{alumno} aprobó con {nota:.2f} puntos'`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

alumno = 'Luciano'
nota = 8.7512
resultado = 

comprobar("py05")

## 6. Limpiar texto
`Básico` · **+10 XP**

Los textos tienen **métodos**: funciones que se escriben después de un punto. Los más usados para limpiar datos:

| Método | Qué hace | Ejemplo |
|---|---|---|
| `.strip()` | quita espacios al inicio y al final | `"  hola ".strip()` → `'hola'` |
| `.lower()` / `.upper()` | minúsculas / mayúsculas | `"Hola".upper()` → `'HOLA'` |
| `.title()` | primera letra de cada palabra en mayúscula | `"buenos aires".title()` → `'Buenos Aires'` |
| `.replace(a, b)` | reemplaza `a` por `b` | `"1.200".replace(".", "")` → `'1200'` |

Los métodos se pueden **encadenar**: se aplican de izquierda a derecha.

```python
"  JUAN pérez ".strip().title()   # 'Juan Pérez'
```

> ⚠️ Los strings son **inmutables**: `.upper()` no cambia la variable original, devuelve un texto nuevo. Por eso se guarda el resultado: `nombre = nombre.upper()`.

### 🎯 Tu misión
Limpia el texto `sucio`: quítale los espacios de los extremos y déjalo con formato de título. Guarda el texto limpio en `resultado`.

<details><summary>💡 Pista</summary>

`sucio.strip().title()`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

sucio = '   sAN miguel de TUCUMÁN  '
resultado = sucio

comprobar("py06")

## 7. Índices y recortes
`Intermedio` · **+20 XP**

Un texto es una **secuencia** de caracteres, y cada uno tiene una posición (índice) que **empieza en 0**:

```
 texto =  P  y  t  h  o  n
 índice:  0  1  2  3  4  5
negativo: -6 -5 -4 -3 -2 -1
```

```python
texto = "Python"
texto[0]      # 'P'    primer carácter
texto[-1]     # 'n'    último carácter
texto[0:3]    # 'Pyt'  del 0 al 3 (el 3 NO se incluye)
texto[2:]     # 'thon' desde el 2 hasta el final
texto[:2]     # 'Py'   desde el inicio hasta el 2
len(texto)    # 6      cantidad de caracteres
```

Esta notación `[inicio:fin]` se llama **slicing** y funciona igual en listas, arrays de NumPy y (con `iloc`) en pandas. Vale la pena dominarla ahora.

### 🎯 Tu misión
El código de producto `'ARG-2024-00731'` tiene el país en los **primeros 3** caracteres y el número de serie en los **últimos 5**. Guarda en `resultado` una lista `[pais, serie]`.

<details><summary>💡 Pista</summary>

`codigo[:3]  y  codigo[-5:]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

codigo = 'ARG-2024-00731'
pais = 
serie = 
resultado = [pais, serie]

comprobar("py07")

## 8. Separar y unir
`Intermedio` · **+20 XP**

Dos métodos que vas a usar todo el tiempo:

- `texto.split(sep)` **corta** un texto en una lista, usando `sep` como separador.
- `sep.join(lista)` hace lo contrario: **une** una lista de textos usando `sep` entre cada uno.

```python
fila = "Ana;28;Córdoba"
partes = fila.split(";")      # ['Ana', '28', 'Córdoba']

palabras = ["ciencia", "de", "datos"]
" ".join(palabras)            # 'ciencia de datos'
"-".join(palabras)            # 'ciencia-de-datos'
```

> 💼 **En el trabajo:** así se procesan líneas de un CSV a mano, se arman rutas o se generan identificadores.

### 🎯 Tu misión
Convierte el email `'luciano.rosales@datos.com'` en el usuario `'luciano_rosales'`: toma la parte **antes** de la `@` y reemplaza el punto por guion bajo usando `split` y `join`.

<details><summary>💡 Pista</summary>

`'_'.join(usuario.split('.'))`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

email = 'luciano.rosales@datos.com'
usuario = email.split('@')[0]    # 'luciano.rosales'
resultado = 

comprobar("py08")

---
# 📌 Listas y tuplas

## 9. Tu primera lista
`Básico` · **+10 XP**

Una **lista** guarda varios valores en orden, entre corchetes `[]` y separados por comas. Puede tener cualquier tipo de dato:

```python
ventas = [120, 95, 143, 88, 170]
ventas[0]        # 120  primer elemento
ventas[-1]       # 170  último
ventas[1:3]      # [95, 143]
len(ventas)      # 5
```

Funciones útiles con listas de números:

```python
sum(ventas)      # 616
max(ventas)      # 170
min(ventas)      # 88
sum(ventas) / len(ventas)   # 123.2 -> promedio
```

> 💡 Las listas son **mutables**: se pueden cambiar después de creadas (`ventas[0] = 130`).

### 🎯 Tu misión
Con la lista `temperaturas`, guarda en `resultado` una lista con tres valores: la **máxima**, la **mínima** y el **promedio** (suma dividida por la cantidad).

<details><summary>💡 Pista</summary>

`[max(temperaturas), min(temperaturas), sum(temperaturas) / len(temperaturas)]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

temperaturas = [18.5, 22.0, 25.5, 19.0, 30.0]
resultado = [ , , ]

comprobar("py09")

## 10. Modificar listas
`Intermedio` · **+20 XP**

Las listas tienen métodos para agregar, quitar y ordenar:

| Método | Qué hace |
|---|---|
| `lista.append(x)` | agrega `x` al final |
| `lista.insert(i, x)` | inserta `x` en la posición `i` |
| `lista.remove(x)` | quita el primer `x` que encuentra |
| `lista.pop()` | quita y devuelve el último |
| `lista.sort()` | ordena la lista (de menor a mayor) |
| `lista.sort(reverse=True)` | ordena de mayor a menor |

```python
tareas = ["limpiar", "analizar"]
tareas.append("graficar")     # ['limpiar', 'analizar', 'graficar']
tareas.insert(0, "cargar")    # ['cargar', 'limpiar', 'analizar', 'graficar']
```

> ⚠️ `lista.sort()` modifica la lista y devuelve `None`. Si escribes `x = lista.sort()`, `x` queda vacío. Para obtener una lista nueva ordenada sin tocar la original usa `sorted(lista)`.

### 🎯 Tu misión
A la lista `notas` agrégale un `9` al final, quítale el `2` (fue un error de carga) y ordénala de **mayor a menor**. Guarda la lista final en `resultado`.

<details><summary>💡 Pista</summary>

`notas.append(9)  notas.remove(2)  notas.sort(reverse=True)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

notas = [7, 2, 10, 6, 8]

resultado = notas

comprobar("py10")

## 11. Tuplas y desempaquetado
`Intermedio` · **+20 XP**

Una **tupla** es como una lista pero **inmutable** (no se puede modificar). Se escribe con paréntesis:

```python
coordenadas = (-31.42, -64.18)   # latitud, longitud de Córdoba
coordenadas[0]                    # -31.42
```

Se usan para datos que van juntos y no deberían cambiar. Y permiten el **desempaquetado**: repartir los valores en varias variables de una sola vez.

```python
lat, lon = coordenadas            # lat = -31.42, lon = -64.18
nombre, edad, ciudad = ("Ana", 28, "Lima")
a, b = b, a                       # truco: intercambia dos variables
```

> 💼 **En el trabajo:** lo vas a ver en todos lados. Por ejemplo, `df.shape` devuelve una tupla `(filas, columnas)` y en ML se escribe `X_train, X_test, y_train, y_test = train_test_split(...)`.

### 🎯 Tu misión
La tupla `dimensiones` tiene `(filas, columnas)` de una tabla. Desempaquétala en las variables `filas` y `columnas`, y guarda en `resultado` la **cantidad total de celdas**.

<details><summary>💡 Pista</summary>

`filas, columnas = dimensiones`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

dimensiones = (1500, 12)

resultado = 

comprobar("py11")

---
# 📌 Diccionarios y conjuntos

## 12. Diccionarios
`Básico` · **+10 XP**

Un **diccionario** guarda pares **clave: valor** entre llaves `{}`. En lugar de buscar por posición, buscas por **clave**:

```python
socio = {"nombre": "Ana", "edad": 28, "plan": "Premium"}
socio["nombre"]            # 'Ana'
socio["edad"] = 29         # modifica un valor
socio["email"] = "a@m.com" # agrega una clave nueva
socio.get("telefono", "sin dato")   # 'sin dato' -> get no da error si falta la clave
list(socio.keys())         # ['nombre', 'edad', 'plan', 'email']
```

> 💼 **En el trabajo:** es la estructura más importante después de la lista. Un JSON de una API es un diccionario, cada fila de una base de datos se puede ver como uno, y `pd.DataFrame(dict)` crea tablas a partir de diccionarios.

### 🎯 Tu misión
Al diccionario `producto`: cambia el `precio` a `1100`, agrega la clave `'stock'` con valor `8`, y guarda en `resultado` el diccionario completo.

<details><summary>💡 Pista</summary>

`producto['precio'] = 1100   y   producto['stock'] = 8`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

producto = {'nombre': 'Monitor', 'precio': 1000}

resultado = producto

comprobar("py12")

## 13. Recorrer un diccionario
`Intermedio` · **+20 XP**

Con `.items()` recorres un diccionario obteniendo **clave y valor** a la vez:

```python
stock = {"mouse": 40, "teclado": 15, "monitor": 3}
for producto, cantidad in stock.items():
    print(f"{producto}: {cantidad}")
```

Otras formas útiles:

```python
sum(stock.values())              # 58 -> suma de todos los valores
max(stock, key=stock.get)        # 'mouse' -> la clave con el valor más alto
"monitor" in stock               # True -> ¿existe esa clave?
```

> 💡 `in` sobre un diccionario busca entre las **claves**, no entre los valores.

### 🎯 Tu misión
`ventas` tiene las ventas de cada vendedor. Guarda en `resultado` una lista con el **total vendido** y el **nombre del vendedor que más vendió**.

<details><summary>💡 Pista</summary>

`[sum(ventas.values()), max(ventas, key=ventas.get)]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

ventas = {'Sofía': 1200, 'Tomás': 950, 'Valeria': 1430, 'Andrés': 870}
resultado = [ , ]

comprobar("py13")

## 14. Conjuntos: valores únicos
`Intermedio` · **+20 XP**

Un **conjunto** (`set`) guarda valores **sin repetir** y sin orden. Es la forma más rápida de eliminar duplicados o comparar grupos:

```python
ciudades = ["Lima", "Quito", "Lima", "Bogotá", "Quito"]
set(ciudades)                    # {'Lima', 'Quito', 'Bogotá'}
len(set(ciudades))               # 3 -> cuántos valores distintos hay

a = {"python", "sql", "excel"}
b = {"python", "sql", "tableau"}
a & b    # {'python', 'sql'}        intersección: están en los dos
a | b    # los 4 sin repetir        unión
a - b    # {'excel'}                están en a pero no en b
```

> 💼 **En el trabajo:** "¿qué clientes compraron en enero pero no en febrero?" se responde con una resta de conjuntos. Es la misma idea que un LEFT JOIN con `IS NULL` en SQL.

### 🎯 Tu misión
Guarda en `resultado` una lista **ordenada alfabéticamente** con los clientes que compraron en `enero` pero **no** en `febrero`.

<details><summary>💡 Pista</summary>

`sorted(set(enero) - set(febrero))`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

enero = ['Ana', 'Bruno', 'Carla', 'Diego', 'Ana']
febrero = ['Bruno', 'Elena', 'Diego']
resultado = 

comprobar("py14")

---
# 📌 Decisiones con if

## 15. if, elif y else
`Básico` · **+10 XP**

`if` ejecuta un bloque de código **solo si** se cumple una condición. `elif` ("si no, si…") agrega más condiciones y `else` cubre todo lo demás:

```python
temperatura = 31
if temperatura > 30:
    estado = "calor"
elif temperatura > 15:
    estado = "templado"
else:
    estado = "frío"
```

Dos cosas clave de Python:
- Después de la condición van **dos puntos** `:`.
- El bloque se marca con **sangría** (4 espacios). La sangría no es decoración: define qué código está dentro del `if`.

Comparaciones: `==` (igual), `!=` (distinto), `>`, `<`, `>=`, `<=`.

> ⚠️ `=` guarda un valor; `==` compara. `if x = 5:` da error.

### 🎯 Tu misión
Clasifica la `nota`: `'Promociona'` si es 8 o más, `'Aprueba'` si es 6 o más, y `'Recupera'` en otro caso. Guarda el texto en `resultado`.

<details><summary>💡 Pista</summary>

`if nota >= 8: ... elif nota >= 6: ... else: ...`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

nota = 7

resultado = 

comprobar("py15")

## 16. Condiciones combinadas
`Intermedio` · **+20 XP**

Para combinar condiciones se usan `and`, `or` y `not`:

| Operador | Es verdadero si… |
|---|---|
| `a and b` | se cumplen **las dos** |
| `a or b` | se cumple **al menos una** |
| `not a` | `a` es falsa |

```python
edad, socio = 25, True
if edad >= 18 and socio:
    print("Puede entrar")

dia = "sáb"
es_finde = dia in ["sáb", "dom"]    # in: ¿está en la lista?
```

> 💡 En Python puedes encadenar comparaciones como en matemática: `18 <= edad <= 65`.

### 🎯 Tu misión
Un cliente recibe descuento si gastó **más de 50000** y es `premium`, **o** si es su `primera_compra`. Guarda en `resultado` el booleano (`True`/`False`) de si recibe descuento.

<details><summary>💡 Pista</summary>

`(gasto > 50000 and premium) or primera_compra`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

gasto = 62000
premium = False
primera_compra = True
resultado = 

comprobar("py16")

---
# 📌 Bucles

## 17. for y range
`Básico` · **+10 XP**

Un bucle `for` **repite** un bloque para cada elemento de una secuencia:

```python
for fruta in ["manzana", "pera", "uva"]:
    print(fruta)
```

`range()` genera secuencias de números:

```python
range(5)          # 0, 1, 2, 3, 4      (el 5 no se incluye)
range(1, 6)       # 1, 2, 3, 4, 5
range(0, 10, 2)   # 0, 2, 4, 6, 8      (de 2 en 2)
```

El patrón **acumulador** es clásico: una variable empieza en 0 y el bucle le va sumando.

```python
total = 0
for n in range(1, 4):
    total = total + n     # o total += n
# total = 6
```

### 🎯 Tu misión
Usando un bucle `for` con `range`, suma los **números pares del 2 al 100** (ambos incluidos). Guarda la suma en `resultado`.

<details><summary>💡 Pista</summary>

`for n in range(2, 101, 2):
    total += n`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

total = 0
# tu bucle aquí 👇

resultado = total

comprobar("py17")

## 18. Recorrer y contar
`Intermedio` · **+20 XP**

Combinando `for` con `if` puedes **filtrar** y **contar** mientras recorres una lista:

```python
edades = [15, 32, 17, 45, 22]
mayores = []
for e in edades:
    if e >= 18:
        mayores.append(e)
# mayores = [32, 45, 22]
```

Dos palabras clave para controlar el bucle:
- `break`: sale del bucle inmediatamente.
- `continue`: salta a la siguiente vuelta.

> 💼 **En el trabajo:** en pandas casi nunca harás esto con bucles (hay formas vectorizadas mucho más rápidas), pero entender el bucle es lo que te permite entender qué hace pandas por dentro.

### 🎯 Tu misión
Recorre `pagos` y guarda en `resultado` una lista con los montos **mayores a 10000**, en el mismo orden.

<details><summary>💡 Pista</summary>

`for p in pagos:
    if p > 10000:
        altos.append(p)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

pagos = [8000, 12000, 15000, 9500, 10000, 11000]
altos = []
# tu bucle aquí 👇

resultado = altos

comprobar("py18")

## 19. while: repetir hasta que…
`Intermedio` · **+20 XP**

`while` repite un bloque **mientras** una condición sea verdadera. Se usa cuando no sabes de antemano cuántas vueltas vas a dar:

```python
saldo = 100
meses = 0
while saldo < 200:
    saldo = saldo * 1.10     # crece un 10% por mes
    meses += 1
# meses = 8
```

> ⚠️ Si la condición nunca se vuelve falsa, el bucle no termina nunca (**bucle infinito**). Asegúrate de que algo dentro del bucle cambie la condición.

### 🎯 Tu misión
Un ahorro de `50000` crece un **5% por mes** (multiplica por `1.05`). Con un `while`, calcula cuántos **meses** hacen falta para llegar a **100000 o más**. Guarda los meses en `resultado`.

<details><summary>💡 Pista</summary>

`while ahorro < 100000:
    ahorro *= 1.05
    meses += 1`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

ahorro = 50000
meses = 0
# tu while aquí 👇

resultado = meses

comprobar("py19")

## 20. enumerate y zip
`Intermedio` · **+20 XP**

Dos funciones que hacen los bucles mucho más limpios:

`enumerate` te da la **posición** y el **valor** a la vez:

```python
for i, nombre in enumerate(["Ana", "Luis"], start=1):
    print(i, nombre)     # 1 Ana / 2 Luis
```

`zip` recorre **varias listas en paralelo**, elemento por elemento:

```python
productos = ["mouse", "teclado"]
precios = [25, 45]
for prod, precio in zip(productos, precios):
    print(f"{prod}: ${precio}")

dict(zip(productos, precios))   # {'mouse': 25, 'teclado': 45}
```

> 💡 Viste `enumerate(zip(...))` en la Clase 4: es combinar las dos ideas.

### 🎯 Tu misión
Usa `zip` para calcular el **ingreso** de cada producto (`precio * cantidad`) y guarda en `resultado` un **diccionario** `{producto: ingreso}`.

<details><summary>💡 Pista</summary>

`for prod, precio, cant in zip(productos, precios, cantidades):
    resultado[prod] = precio * cant`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

productos = ['Laptop', 'Mouse', 'Monitor']
precios = [1200, 25, 300]
cantidades = [3, 40, 5]
resultado = {}

comprobar("py20")

## 21. List comprehensions
`Avanzado` · **+35 XP**

Una **list comprehension** crea una lista nueva en **una sola línea**. Es uno de los rasgos más característicos de Python, y lo vas a leer en cualquier código profesional:

```python
# forma larga
cuadrados = []
for n in range(5):
    cuadrados.append(n ** 2)

# list comprehension: lo mismo en una línea
cuadrados = [n ** 2 for n in range(5)]        # [0, 1, 4, 9, 16]

# con filtro: solo los pares
pares = [n for n in range(10) if n % 2 == 0]  # [0, 2, 4, 6, 8]
```

Se lee así: "**dame** `expresión` **para cada** `elemento` **en** `secuencia` **si** `condición`".

También existen para diccionarios: `{k: v for k, v in pares}`.

### 🎯 Tu misión
Con una **list comprehension**, crea una lista con los precios **con 21% de IVA** (redondeados a 2 decimales con `round(x, 2)`) pero **solo** de los precios mayores a 50. Guárdala en `resultado`.

<details><summary>💡 Pista</summary>

`[round(p * 1.21, 2) for p in precios if p > 50]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

precios = [100, 25.5, 80, 45, 300]
resultado = 

comprobar("py21")

---
# 📌 Funciones

## 22. Tu primera función
`Básico` · **+10 XP**

Una **función** es un bloque de código con nombre que puedes reutilizar. Se define con `def`, recibe **parámetros** y devuelve un valor con `return`:

```python
def area_rectangulo(base, altura):
    return base * altura

area_rectangulo(3, 4)    # 12
area_rectangulo(10, 2)   # 20
```

Buenas prácticas:
- Nombres con verbo o descriptivos: `calcular_total`, `limpiar_texto`.
- Una función hace **una sola cosa**.
- Un comentario entre triples comillas justo debajo del `def` (*docstring*) explica qué hace.

> ⚠️ `print` muestra un valor, pero **no lo devuelve**. Si tu función usa `print` en vez de `return`, el resultado no se puede guardar ni reutilizar.

### 🎯 Tu misión
Define la función `precio_final(precio, descuento)` que devuelva el precio aplicando el descuento (el descuento viene en porcentaje: `20` significa 20%). Después guarda en `resultado` el valor de `precio_final(15000, 20)`.

<details><summary>💡 Pista</summary>

`return precio * (1 - descuento / 100)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

def precio_final(precio, descuento):
    # tu código aquí 👇
    pass

resultado = precio_final(15000, 20)

comprobar("py22")

## 23. Parámetros por defecto
`Intermedio` · **+20 XP**

Un parámetro puede tener un **valor por defecto**: si al llamar la función no lo pasas, usa ese valor.

```python
def saludar(nombre, saludo="Hola"):
    return f"{saludo}, {nombre}!"

saludar("Ana")                    # 'Hola, Ana!'
saludar("Ana", "Buenas")          # 'Buenas, Ana!'
saludar(nombre="Ana", saludo="Hey")   # argumentos por nombre (keyword)
```

Así funcionan casi todas las funciones de pandas: `df.sort_values("col", ascending=False)` usa `ascending=False` para cambiar el valor por defecto (`True`).

> 💡 Los parámetros con valor por defecto van **siempre al final**.

### 🎯 Tu misión
Define `convertir(monto, cotizacion=1000)` que devuelva el monto en pesos dividido por la cotización, **redondeado a 2 decimales**. Guarda en `resultado` una lista con `convertir(250000)` y `convertir(250000, cotizacion=1250)`.

<details><summary>💡 Pista</summary>

`def convertir(monto, cotizacion=1000):
    return round(monto / cotizacion, 2)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

def convertir(monto, cotizacion):
    pass

resultado = [convertir(250000), convertir(250000, cotizacion=1250)]

comprobar("py23")

## 24. lambda y ordenar con key
`Avanzado` · **+35 XP**

Una **lambda** es una función pequeña y sin nombre, escrita en una línea. Se usa para pasarle una función a otra función:

```python
doble = lambda x: x * 2        # igual que def doble(x): return x * 2
doble(5)                       # 10
```

Su uso más común es el parámetro `key` de `sorted`, `max` y `min`, que indica **por qué** ordenar:

```python
alumnos = [("Ana", 8), ("Luis", 9), ("Eva", 7)]
sorted(alumnos, key=lambda a: a[1])                 # por nota, de menor a mayor
sorted(alumnos, key=lambda a: a[1], reverse=True)   # de mayor a menor
```

> 💼 **En el trabajo:** vas a ver lambdas en pandas todo el tiempo: `df["col"].apply(lambda x: x.strip())` o `df.sort_values(key=...)`.

### 🎯 Tu misión
Ordena la lista `productos` (tuplas de `(nombre, precio)`) por **precio de mayor a menor** usando `sorted` con una `lambda`. Guarda en `resultado` solo los **nombres** en ese orden.

<details><summary>💡 Pista</summary>

`sorted(productos, key=lambda p: p[1], reverse=True)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

productos = [('Mouse', 25), ('Laptop', 1200), ('Monitor', 300), ('Teclado', 45)]
ordenados = 
resultado = [p[0] for p in ordenados]

comprobar("py24")

---
# 📌 Código profesional

## 25. Manejar errores con try
`Intermedio` · **+20 XP**

Cuando algo puede fallar (un dato mal cargado, un archivo que no existe), `try / except` evita que el programa se detenga:

```python
def a_numero(texto):
    try:
        return float(texto)
    except ValueError:        # solo atrapa ese tipo de error
        return None

a_numero("12.5")    # 12.5
a_numero("n/d")     # None, sin romper el programa
```

Errores que vas a ver seguido:

| Error | Cuándo aparece |
|---|---|
| `ValueError` | un valor con formato inválido (`int("abc")`) |
| `KeyError` | una clave o columna que no existe |
| `TypeError` | operar tipos incompatibles (`"5" + 3`) |
| `ZeroDivisionError` | dividir por cero |

> ⚠️ Evita `except:` a secas: atrapa **todos** los errores y esconde problemas reales. Indica siempre qué error esperas.

### 🎯 Tu misión
Recorre `edades_texto`, convierte cada valor con `int()`, y **saltea** los que no se pueden convertir (atrapa el `ValueError`). Guarda en `resultado` la lista de edades válidas.

<details><summary>💡 Pista</summary>

`try:
    resultado.append(int(e))
except ValueError:
    pass`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

edades_texto = ['34', '28', 'n/d', '45', '', '19']
resultado = []
for e in edades_texto:
    resultado.append(int(e))   # ⚠️ se rompe con 'n/d'

comprobar("py25")

## 26. Importar módulos
`Básico` · **+10 XP**

Python trae una **biblioteca estándar** enorme, y además hay miles de librerías externas (pandas, NumPy…). Para usarlas se **importan**:

```python
import math                       # importa el módulo completo
math.sqrt(16)                     # 4.0

from statistics import median     # importa solo una función
median([3, 1, 2])                 # 2

import pandas as pd               # importa con un alias (apodo)
```

Módulos estándar muy útiles: `math` (matemática), `statistics` (estadística básica), `random` (azar), `datetime` (fechas), `os` y `pathlib` (archivos), `json` (datos de APIs).

> 💼 **En el trabajo:** los alias son convenciones universales. Siempre `import pandas as pd`, `import numpy as np`, `import matplotlib.pyplot as plt` y `import seaborn as sns`. Si usas otro nombre, tu código va a confundir a todos.

### 🎯 Tu misión
Importa el módulo `statistics` y guarda en `resultado` una lista con la **media** (`mean`), la **mediana** (`median`) y la **moda** (`mode`) de `sueldos`.

<details><summary>💡 Pista</summary>

`import statistics  ->  statistics.mean(sueldos), statistics.median(sueldos), statistics.mode(sueldos)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

sueldos = [30000, 35000, 40000, 45000, 45000, 55000, 500000]

resultado = [ , , ]

comprobar("py26")

## 27. Clases y objetos
`Avanzado` · **+35 XP**

Una **clase** es un molde para crear **objetos** que juntan datos (**atributos**) y comportamiento (**métodos**). Todo en Python es un objeto: un DataFrame es un objeto de la clase `DataFrame`, y `df.head()` es uno de sus métodos.

```python
class Socio:
    def __init__(self, nombre, cuota):   # se ejecuta al crear el objeto
        self.nombre = nombre             # self = el propio objeto
        self.cuota = cuota
        self.pagos = []

    def pagar(self, monto):              # un método
        self.pagos.append(monto)

    def al_dia(self):
        return sum(self.pagos) >= self.cuota

ana = Socio("Ana", 12000)   # crear un objeto (instancia)
ana.pagar(12000)
ana.al_dia()                # True
```

> 💼 **En el trabajo:** no hace falta que escribas muchas clases para analizar datos, pero **necesitas leerlas**: scikit-learn funciona así (`modelo = LinearRegression()` y después `modelo.fit(...)`).

### 🎯 Tu misión
Completa la clase `Cuenta`: el método `depositar(monto)` suma el monto al `saldo`, y `retirar(monto)` lo resta **solo si hay saldo suficiente** (si no, no hace nada). Crea una cuenta con saldo `1000`, deposita `500`, intenta retirar `2000` y después retira `300`. Guarda el saldo final en `resultado`.

<details><summary>💡 Pista</summary>

`def retirar(self, monto):
    if monto <= self.saldo:
        self.saldo -= monto`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

class Cuenta:
    def __init__(self, saldo):
        self.saldo = saldo

    def depositar(self, monto):
        pass

    def retirar(self, monto):
        pass

cuenta = Cuenta(1000)
cuenta.depositar(500)
cuenta.retirar(2000)
cuenta.retirar(300)
resultado = cuenta.saldo

comprobar("py27")

---
# 🏁 ¡Módulo completado!
Vuelve a [PandasQuest](https://rosalesluciano.github.io/pandas-quest/) para sumar tus XP y seguir con el siguiente módulo. 🚀